<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Step 4</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Step 4: Skills, the only thing you add to grow the agent</h1>
<p style="margin:.55em 0">A skill takes <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">ctx</code> (the conversation) and typed arguments, and returns a dict. It explains failures as data
with a <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">problem</code> and often a <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">next</code>: the model reads <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">next</code> right when it matters, which
works far better than a rule buried in the prompt.</p>

</div>

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Codes and SMS first</h2>
<p style="margin:.55em 0">A one-time code is texted, never shown to the model; it expires, locks after wrong tries, and only a few can be sent
per call (stops "SMS pumping"). The SMS sender is an interface: the simulated one keeps an outbox for the page.</p>

</div>

In [1]:
%%writefile build/app/verification.py
"""One-time codes and the SMS that carries them.

The code never reaches the model: Otp texts it and checks it; tools only ever see "code sent to phone ending 2367".
Production: implement Sms.send with a real provider (Twilio, MSG91, ...) and pick it in app.session.
"""
import hmac
import logging
import re
import secrets
import time
from typing import Callable, Protocol

log = logging.getLogger(__name__)


class Sms(Protocol):
    def send(self, to: str, text: str) -> None: ...


class SimulatedSms:
    """Development/demo SMS: keeps an outbox and hands every message to `on_send` (the page's phone panel)."""

    def __init__(self, on_send: Callable[[dict], None] = lambda m: None):
        self.outbox: list[dict] = []
        self.on_send = on_send

    def send(self, to: str, text: str) -> None:
        msg = {"to": to, "text": text}
        self.outbox.append(msg)
        self.on_send(msg)

    def last_code(self) -> str:
        """Test helper: the digits of the latest SMS (what a patient reads off their phone)."""
        return re.sub(r"\D", "", self.outbox[-1]["text"]) if self.outbox else ""


class Otp:
    """One pending code per conversation. A new code replaces the old; codes are single-use, expire after `ttl`
    seconds, lock after `max_tries` wrong answers, and at most `max_sends` are texted per conversation."""

    def __init__(self, sms: Sms, sender: str = "the clinic", ttl: float = 300, max_tries: int = 3, max_sends: int = 3,
                 gen: Callable[[], str] = lambda: f"{secrets.randbelow(10 ** 6):06d}", clock=time.monotonic):
        self.sms, self.sender, self.ttl, self.max_tries, self.max_sends = sms, sender, ttl, max_tries, max_sends
        self.gen, self.clock = gen, clock
        self.sent = 0
        self.pending = None  # {"purpose", "payload", "code", "at", "tries"}

    def send(self, phone: str, purpose: str, payload) -> dict | None:
        """Text a code. Returns a problem dict instead when the per-conversation limit is reached."""
        if self.sent >= self.max_sends:
            log.warning("otp send limit reached (%d)", self.max_sends)
            return {"status": "too_many_codes", "problem": "too many codes were sent in this call",
                    "next": "Apologise and suggest calling again later."}
        code = self.gen()
        self.pending = {"purpose": purpose, "payload": payload, "code": code, "at": self.clock(), "tries": 0}
        self.sent += 1
        self.sms.send("+" + phone, f"Your {self.sender} {purpose} code is {code}.")
        return None

    def check(self, code) -> tuple[str | None, object, dict | None]:
        """(purpose, payload, None) on a match, else (None, None, problem-for-the-model)."""
        p = self.pending
        if not p:
            return None, None, {"status": "no_code_pending", "problem": "no code was sent in this conversation"}
        if self.clock() - p["at"] > self.ttl:
            self.pending = None
            return None, None, {"status": "expired", "problem": "the code expired; start again"}
        if not hmac.compare_digest(re.sub(r"\D", "", str(code)), p["code"]):
            p["tries"] += 1
            if p["tries"] >= self.max_tries:
                self.pending = None
                return None, None, {"status": "locked", "problem": "too many wrong codes; start again"}
            return None, None, {"status": "wrong_code", "tries_left": self.max_tries - p["tries"]}
        self.pending = None
        return p["purpose"], p["payload"], None


Writing build/app/verification.py


In [2]:
%%writefile build/app/skills/__init__.py
"""Skill modules. Every module here is found by app.registry.discover(); its file name is its toolset."""


Writing build/app/skills/__init__.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Identity</h2>
<p style="margin:.55em 0">Verify a phone by code; who is on the phone is revealed only after the code proves the caller holds it. A shared phone
(a parent and child) asks which person. A placeholder instead of a code never burns a try.</p>

</div>

In [3]:
%%writefile build/app/skills/identity.py
"""Who is calling: verify a phone with a texted code, then pick the patient if several share it."""
import re

from app.domain import patients
from app.registry import skill


def verified(ctx, p: dict) -> dict:
    ctx.patient, ctx.choices = {"id": p["id"], "name": p["name"]}, None
    return {"status": "verified", "name": p["name"],
            "next": f"Greet {p['name'].split()[0]}, then carry on with what they already asked for "
                    "(for example call my_appointments if they wanted their appointments). "
                    "Ask only if they asked nothing yet."}


@skill
def verify_patient(ctx, phone: str) -> dict:
    """Look up an existing patient by mobile number and text them a 6-digit code to prove it is their phone.

    When: first step for any existing patient; call it the moment they give a mobile number.
    Args:
        phone: 10-digit mobile number, digit for digit as the caller said it
    """
    number = patients.normalize_phone(phone)
    if not number:
        return {"status": "bad_phone", "problem": "need a 10-digit mobile number"}
    found = patients.by_phone(ctx.db, number)
    if not found:
        ctx.otp.pending = None
        return {"status": "not_found", "problem": "no patient has this number",
                "next": "Check the number, or offer to register them as a new patient."}
    # who is on this phone is only revealed after the code proves the caller holds it
    if problem := ctx.otp.send(number, "verification", found):
        return problem
    return {"status": "code_sent", "sent_to": f"phone ending {number[-4:]}",
            "next": "Ask for the 6-digit code that was just texted."}


Writing build/app/skills/identity.py


In [4]:
%%writefile -a build/app/skills/identity.py

@skill
def confirm_code(ctx, code: str) -> dict:
    """Check the 6-digit code the caller read from the SMS (for verification or registration).

    When: only after the caller has read out the 6 digits of a code.
    Args:
        code: exactly the 6 digits the caller read out
    """
    if len(re.sub(r"\D", "", code)) != 6:  # a placeholder or a half-heard code must not burn a try
        return {"status": "need_digits", "problem": "that is not a 6-digit code",
                "next": "Ask the caller to read the 6 digits from the SMS." if ctx.otp.pending else
                        "No code has been sent yet: call verify_patient (or start_registration) first."}
    if not ctx.otp.pending and ctx.patient:  # e.g. the model repeats a call after a reconnect
        return {"status": "already_verified", "name": ctx.patient["name"],
                "next": "They are verified: carry on with what they asked for."}
    purpose, payload, problem = ctx.otp.check(code)
    if problem:
        return problem
    if purpose == "registration":
        out = patients.insert(ctx.db, payload)
        if out["status"] == "registered":
            verified(ctx, {"id": out["patient_id"], "name": out["name"]})
            out["next"] = "Give them their patient ID, then ask if they would like to book an appointment."
        return out
    if len(payload) == 1:
        return verified(ctx, payload[0])
    ctx.choices = payload
    return {"status": "choose_patient", "names": [p["name"] for p in payload],
            "next": "Several patients share this phone: ask which of them they are calling about."}


@skill
def choose_patient(ctx, name: str) -> dict:
    """Pick which patient on a shared phone the caller means, after confirm_code asked to choose.

    When: after confirm_code returns choose_patient.
    Args:
        name: the name the caller said
    """
    if not ctx.choices:
        return {"status": "nothing_to_choose", "problem": "verify the phone first"}
    said = name.lower().split()
    match = [p for p in ctx.choices if said and all(w in p["name"].lower().split() for w in said)]
    if len(match) != 1:
        return {"status": "which_one", "names": [p["name"] for p in ctx.choices]}
    return verified(ctx, match[0])


Appending to build/app/skills/identity.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Registration</h2><p style="margin:.55em 0">Validate, check duplicates, text a code. The same <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">confirm_code</code> finishes it: one door for every code.</p>

</div>

In [5]:
%%writefile build/app/skills/registration.py
"""New patients: validate, check duplicates, text a code. confirm_code (identity) writes the record."""
from typing import Literal

from app.domain import patients
from app.registry import skill


@skill
def start_registration(ctx, first_name: str, last_name: str, phone: str, sex: Literal["Male", "Female"],
                       dob: str = "", age: str = "", email: str = "") -> dict:
    """Validate a NEW patient's details and text a 6-digit code to their phone. Call again with corrections.

    When: for new patients: call it the moment you have first and last name, mobile, sex and date of birth (or age), in that same turn; its result is what you read back. Do not ask for email.
    Args:
        phone: 10-digit mobile number, digits only
        dob: YYYY-MM-DD, or empty if only age is known
        age: age in years, or empty if dob is given
        email: empty if not given
    """
    rec, errs = patients.validate(first_name, last_name, phone, sex, dob, age, email)
    if not errs:
        errs = patients.check_new(ctx.db, rec)
    if errs:
        ctx.otp.pending = None  # a failed attempt also cancels an older pending code
        return {"status": "invalid", "problems": errs}
    if problem := ctx.otp.send(rec["phone"], "registration", rec):
        return problem
    return {"status": "code_sent", "sent_to": f"phone ending {rec['phone'][-4:]}", "read_back": patients.read_back(rec),
            "next": "First read back every detail in read_back: spell both names letter by letter and say "
                    "the phone digits one by one. Then ask for the 6-digit code that was just texted."}


Writing build/app/skills/registration.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Appointments</h2><p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">needs_patient=True</code> is the guard. Each skill is a thin wrapper over the rules from step 3.</p>

</div>

In [6]:
%%writefile build/app/skills/appointments.py
"""Appointments for the verified patient: free slots, book, list, cancel."""
from app.domain import appointments as rules
from app.registry import skill


@skill
def find_slots(ctx, doctor: str, date: str) -> dict:
    """Free appointment times for one doctor on one date.

    When: before booking, to offer times.
    Args:
        doctor: doctor's name or specialty in English letters, like Kapoor or pediatrician
        date: YYYY-MM-DD
    """
    return rules.find_slots(ctx.db, doctor, date, ctx.now())


@skill(needs_patient=True)
def book_appointment(ctx, doctor: str, date: str, time: str, reason: str) -> dict:
    """Book an appointment for the verified patient.

    When: after the caller said yes to a doctor, day, time and reason.
    Args:
        doctor: doctor's name or specialty in English letters, like Kapoor or pediatrician
        date: YYYY-MM-DD
        time: a free time from find_slots, like 9:15 AM
        reason: what the visit is for, in a few words
    """
    return rules.book(ctx.db, ctx.patient["id"], doctor, date, time, reason, ctx.now())


@skill(needs_patient=True)
def my_appointments(ctx) -> dict:
    """The verified patient's upcoming appointments.

    When: when the caller asks about their appointments.
    """
    found = rules.upcoming(ctx.db, ctx.patient["id"], ctx.now())
    if found:
        return {"patient": ctx.patient["name"], "appointments": found}
    return {"patient": ctx.patient["name"], "appointments": [], "next": "Say there are none, and offer to book one."}


@skill(needs_patient=True)
def cancel_appointment(ctx, appointment_id: int) -> dict:
    """Cancel one of the verified patient's appointments.

    When: after the caller said yes to cancelling a specific appointment.
    Args:
        appointment_id: from my_appointments
    """
    return rules.cancel(ctx.db, ctx.patient["id"], appointment_id, ctx.now())


Writing build/app/skills/appointments.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Public information, and a second agent's tools</h2>

</div>

In [7]:
%%writefile build/app/skills/clinic_info.py
"""Public clinic information: no patient needed."""
from app.domain import appointments as rules
from app.registry import skill


@skill
def list_doctors(ctx) -> dict:
    """List the clinic's doctors, their specialties and working days.

    When: when asked which doctors there are.
    """
    return {"doctors": rules.doctors(ctx.db)}


@skill
def clinic_hours(ctx) -> dict:
    """The clinic's opening hours.

    When: when asked when the clinic is open.
    """
    return {"hours": "Monday to Friday, 9 AM to 5 PM. Closed on weekends."}


Writing build/app/skills/clinic_info.py


In [8]:
%%writefile build/app/skills/orders.py
"""Support demo (notebook 05): order status and a unit conversion. Used by the "support" agent."""
from typing import Literal

from app.registry import skill

ORDERS = {"A123": {"status": "shipped", "arrives": "Thursday"},
          "B456": {"status": "processing", "arrives": "next week"}}


@skill
def get_order_status(ctx, order_id: str) -> dict:
    """Look up a customer's order by its ID, e.g. A123.

    When: when asked about an order.
    Args:
        order_id: Order ID like A123
    """
    key = "".join(ch for ch in order_id.upper() if ch.isalnum())
    if key not in ORDERS:
        return {"status": "not_found", "problem": f"no order {key}"}
    return {"status": "ok", **ORDERS[key]}


@skill
def convert_temperature(ctx, value: float, to: Literal["C", "F"]) -> dict:
    """Convert a temperature to Celsius (C) or Fahrenheit (F).

    When: when asked to convert a temperature.
    """
    result = round(value * 9 / 5 + 32, 1) if to == "F" else round((value - 32) * 5 / 9, 1)
    return {"status": "ok", "result": result, "unit": to}


Writing build/app/skills/orders.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Try it</h2><p style="margin:.55em 0">Discovery finds every module; each becomes a toolset:</p>

</div>

In [9]:
import sys
sys.path.insert(0, 'build')   # `import app` now means the app you are building

In [10]:
from app import registry
registry.discover()

{'appointments': ['find_slots',
  'book_appointment',
  'my_appointments',
  'cancel_appointment'],
 'clinic_info': ['list_doctors', 'clinic_hours'],
 'identity': ['verify_patient', 'confirm_code', 'choose_patient'],
 'orders': ['get_order_status', 'convert_temperature'],
 'registration': ['start_registration']}

In [11]:
registry.REGISTRY['book_appointment'].spec

{'type': 'function',
 'name': 'book_appointment',
 'description': 'Book an appointment for the verified patient.',
 'parameters': {'type': 'object',
  'properties': {'doctor': {'type': 'string',
    'description': "doctor's name or specialty in English letters, like Kapoor or pediatrician"},
   'date': {'type': 'string', 'description': 'YYYY-MM-DD'},
   'time': {'type': 'string',
    'description': 'a free time from find_slots, like 9:15 AM'},
   'reason': {'type': 'string',
    'description': 'what the visit is for, in a few words'}},
  'required': ['doctor', 'date', 'time', 'reason'],
  'additionalProperties': False}}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Adding a capability</h2>
<p style="margin:.55em 0">A new file in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">app/skills/</code> is all it takes. Here it goes in a separate package so the build stays clean:</p>

</div>

In [12]:
from pathlib import Path
Path('build/extra_skills').mkdir(exist_ok=True)
Path('build/extra_skills/__init__.py').write_text('')
Path('build/extra_skills/pharmacy.py').write_text('''
from app.registry import skill

@skill
def pharmacy_hours(ctx) -> dict:
    """When the pharmacy is open.

    When: when asked about the pharmacy.
    """
    return {"hours": "8 AM to 8 PM"}
''')
registry.discover('extra_skills')['pharmacy']

['pharmacy_hours']

In [13]:
registry.REGISTRY.pop('pharmacy_hours')
import shutil; shutil.rmtree('build/extra_skills')

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><b>Next:</b> <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">05_session.ipynb</code>: one conversation's state and prompt.</p>

</div>